**Q1. (a) Use total least squares to fit the lines to only using the data corresponding to the first line. Report the resulting parameters.**

In [ ]:
import numpy as np


D = np.genfromtxt("lines.csv", delimiter=",", skip_header=1)

# Extract ONLY first line (x1, y1)
x = D[:, 0]
y = D[:, 3]

def total_least_squares(x, y):
    # Step 1: Compute centroid
    x_mean = np.mean(x)
    y_mean = np.mean(y)

    # Step 2: Center the data
    X = np.vstack((x - x_mean, y - y_mean)).T

    # Step 3: Compute covariance matrix (U^T U)
    S = X.T @ X

    # Step 4: Eigen decomposition
    eigenvalues, eigenvectors = np.linalg.eig(S)

    # Step 5: Smallest eigenvalue -> normal vector
    idx = np.argmin(eigenvalues)
    a, b = eigenvectors[:, idx]

    # Normalize (optional but good practice)
    norm = np.sqrt(a**2 + b**2)
    a, b = a / norm, b / norm

    # Step 6: Compute d
    d = a * x_mean + b * y_mean

    return a, b, d

a, b, d = total_least_squares(x, y)

print("Line Parameters")
print(f"a = {a}, b = {b}, d = {d}")
print(f"Line equation: {a}x + {b}y = {d}")

Line Parameters
a = -0.7735616496467873, b = 0.6337210539312553, d = -3.794192210845812
Line equation: -0.7735616496467873x + 0.6337210539312553y = -3.794192210845812


In [ ]:
import numpy as np


D = np.genfromtxt("lines.csv", delimiter=",", skip_header=1)


X_cols = D[:, :3]
Y_cols = D[:, 3:]

X_all = X_cols.flatten()
Y_all = Y_cols.flatten()

points = np.vstack((X_all, Y_all)).T




def fit_line_from_points(p1, p2):
    # Fit line ax + by = d from 2 points

    x1, y1 = p1
    x2, y2 = p2

    # Line normal vector (a, b)
    a = y2 - y1
    b = -(x2 - x1)

    # Normalize
    norm = np.sqrt(a**2 + b**2)
    if norm == 0:
        return None

    a, b = a / norm, b / norm
    d = a * x1 + b * y1

    return a, b, d


def compute_distance(a, b, d, points):
    # Perpendicular distance from points to line

    return np.abs(a * points[:, 0] + b * points[:, 1] - d)


def ransac_line(points, num_iterations=1000, threshold=0.2, min_inliers=20):
    best_model = None
    best_inliers = []

    n_points = len(points)

    for _ in range(num_iterations):
        # Randomly sample 2 points
        idx = np.random.choice(n_points, 2, replace=False)
        p1, p2 = points[idx]

        model = fit_line_from_points(p1, p2)
        if model is None:
            continue

        a, b, d = model

        # Compute distances
        distances = compute_distance(a, b, d, points)

        # Find inliers
        inliers = points[distances < threshold]

        if len(inliers) > len(best_inliers):
            best_inliers = inliers
            best_model = (a, b, d)

    # Refit using all inliers (TLS for better accuracy)
    if best_model is not None and len(best_inliers) >= min_inliers:
        x = best_inliers[:, 0]
        y = best_inliers[:, 1]
        a, b, d = total_least_squares(x, y)
        return (a, b, d), best_inliers

    return None, None


# ---------- Find 3 lines ----------

remaining_points = points.copy()
lines = []

for i in range(3):
    model, inliers = ransac_line(remaining_points)

    if model is None:
        break

    lines.append(model)

    # Remove inliers for next iteration
    mask = np.ones(len(remaining_points), dtype=bool)
    for pt in inliers:
        mask &= ~np.all(remaining_points == pt, axis=1)

    remaining_points = remaining_points[mask]


# ---------- Print Results ----------

print("\nRANSAC detected lines")

for i, (a, b, d) in enumerate(lines):
    print(f"\nLine {i+1}:")
    print(f"a = {a}, b = {b}, d = {d}")
    print(f"Equation: {a}x + {b}y = {d}")


RANSAC detected lines

Line 1:
a = -0.7184167043119765, b = 0.6956129951097221, d = 0.638702282340126
Equation: -0.7184167043119765x + 0.6956129951097221y = 0.638702282340126

Line 2:
a = 0.46489695627333794, b = 0.8853647948996991, d = 1.933771598674861
Equation: 0.46489695627333794x + 0.8853647948996991y = 1.933771598674861

Line 3:
a = -0.770255142602224, b = 0.6377358507209921, d = -3.809491294646967
Equation: -0.770255142602224x + 0.6377358507209921y = -3.809491294646967
